# E00 — Setup and smoke test

| | |
|---|---|
| **Question** | Does the toolchain work in `.venv`, and does Vessim conserve energy on the reference edge site at every step and over the day? |
| **Hypotheses** | H4 (sound energy-side abstractions), H7 (speed, first indication) |
| **Branch** | `exp/e01-energy-equivalence` (E00 delivered together with E01) |
| **Reference site** | Edge study S1.2 inputs (cached in `data/`), 500 kWh battery, one day |

**Checks**
1. Toolchain: package versions, helper package, cached data, edge study environment.
2. Power balance residual at every step.
3. Daily energy closure: supply (PV + grid import + battery discharge) = use (demand + grid export + battery charge), and battery losses = throughput − change of stored energy.
4. Vessim's built-in recorder metrics vs independent integration of the recorded time series.
5. Robustness to the step size (15, 5, 1 min) and wall time.

Two cases: the **reference site** (S1.2: PV always below demand, so the battery only discharges) and a
synthetic **stress case** (IT and cooling load at 10 %, PV x3) that forces battery charging, a full
battery and grid export — paths the reference never exercises.

## 1. Setup

In [1]:
import sys
import importlib.metadata as md_
import pandas as pd
from IPython.display import display

from vessim_eval import DATA_DIR, RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.edge_reference import EDGE_PYTHON, load_reference, scenario_config
from vessim_eval.scenarios import (BatteryParams, SiteInputs, energy_closure, recorder_check,
                                   run_site)

pd.DataFrame({
    "Version": [sys.version.split()[0]] + [md_.version(p) for p in
                ("vessim", "mosaik", "numpy", "pandas", "plotly", "seaborn")],
}, index=["Python", "vessim", "mosaik", "numpy", "pandas", "plotly", "seaborn"])

,Version
Python,3.13.5
vessim,0.16.0
mosaik,3.7.1
numpy,2.5.4
pandas,3.1.1
plotly,7.1.0
seaborn,0.13.2


In [2]:
checks = {
    "Running inside .venv": ".venv" in sys.prefix,
    "Reference data present": (DATA_DIR / "e01_edge_s1_2_15min.csv").exists(),
    "Edge study environment present": EDGE_PYTHON.exists(),
}
pd.Series(checks, name="OK").to_frame()

,OK
Running inside .venv,True
Reference data present,True
Edge study environment present,True


In [3]:
# ---- Parameters -------------------------------------------------------------
SCENARIO = "s1_2_battery_pv_large"
STEP_SIZES_S = [900, 300, 60]          # 15, 5 and 1 min
BATTERY_MODELS = {"Vessim SimpleBattery": "simple", "EfficiencyBattery (losses)": "losses"}
STRESS_LOAD_FACTOR, STRESS_PV_FACTOR = 0.10, 3.0
OUT = RESULTS_DIR / "E00"

site_cfg = scenario_config(SCENARIO)["data_centers"][0]
battery = BatteryParams.from_edge_config(site_cfg["battery_config"])
base_load_kw = site_cfg["hardware"]["base_load"]
initial_stored_kwh = battery.capacity_kwh * battery.soc_init

ref = load_reference("e01_edge_s1_2_15min.csv")
cases = {
    "Reference site": SiteInputs.from_reference(ref, base_load_kw),
    "Stress case": SiteInputs(ref.it_kw * STRESS_LOAD_FACTOR, ref.building_kw,
                              ref.cooling_kw * STRESS_LOAD_FACTOR, ref.pv_kw * STRESS_PV_FACTOR,
                              hardware_base_load_kw=base_load_kw * STRESS_LOAD_FACTOR),
}

In [4]:
fig = vp.stacked_panels(
    [("Site demand (kW)", {n: c.it_kw + c.cooling_kw + c.building_kw for n, c in cases.items()}),
     ("Photovoltaic generation (kW)", {n: c.pv_kw for n, c in cases.items()})],
    title="Inputs of the two cases (15-min samples)")
fig.show()

## 2. Run

In [5]:
runs, rows = {}, []
for case, inputs in cases.items():
    for model_name, model in BATTERY_MODELS.items():
        for step in STEP_SIZES_S:
            key = (case, model_name, step)
            runs[key], wall = run_site(inputs, battery, battery_model=model, control="greedy",
                                       step_s=step)
            rows.append({"Case": case, "Battery model": model_name, "Step (min)": step // 60,
                         "Steps": len(runs[key]), "Wall time (ms)": wall * 1e3})

# One run with Vessim's on-disk experiment record (for the viewer / later comparison).
_, _ = run_site(cases["Reference site"], battery, recorder_output=OUT / "record")
walls = pd.DataFrame(rows)
walls.style.format({"Wall time (ms)": "{:.1f}"}).hide(axis="index")

Case,Battery model,Step (min),Steps,Wall time (ms)
Reference site,Vessim SimpleBattery,15,96,6.4
Reference site,Vessim SimpleBattery,5,288,25.5
Reference site,Vessim SimpleBattery,1,1440,76.1
Reference site,EfficiencyBattery (losses),15,96,5.7
Reference site,EfficiencyBattery (losses),5,288,16.8
Reference site,EfficiencyBattery (losses),1,1440,80.2
Stress case,Vessim SimpleBattery,15,96,5.4
Stress case,Vessim SimpleBattery,5,288,14.9
Stress case,Vessim SimpleBattery,1,1440,116.3
Stress case,EfficiencyBattery (losses),15,96,9.7


## 3. Results

### 3.1 Power balance at every step

In [6]:
resid = pd.Series({k: r["balance_residual_kw"].abs().max() for k, r in runs.items()},
                  name="Maximum power balance residual (kW)")
resid.index.names = ["Case", "Battery model", "Step (s)"]
resid.to_frame().style.format("{:.1e}")

### 3.2 Daily energy closure

In [7]:
closure = pd.DataFrame({k: energy_closure(r, initial_stored_kwh) for k, r in runs.items()}).T
closure.index.names = ["Case", "Battery model", "Step (s)"]
vp.format_table(closure, 2)

In [8]:
from plotly.subplots import make_subplots
for case in cases:
    key = (case, "EfficiencyBattery (losses)", 300)
    fig = vp.energy_balance_bars(closure.loc[key],
                                 title=f"Daily energy balance, {case.lower()} (5-min steps, battery with losses)")
    fig.show()
    vp.save(fig, OUT / f"fig_energy_balance_{case.split()[0].lower()}")

### 3.3 Built-in recorder metrics vs own integration

In [9]:
chk = recorder_check(runs[("Stress case", "EfficiencyBattery (losses)", 300)])
chk.style.format("{:.6g}")

,Vessim recorder,Own integration,Difference
grid_import,7.72363e+06,7.72363e+06,0
grid_export,5.86137e+06,5.86137e+06,-2.79397e-09
energy_produced,9.6807e+06,9.6807e+06,-1.86265e-09
energy_consumed,1.16707e+07,1.16707e+07,-1.86265e-08
peak_grid_import,638802,638802,0


### 3.4 Power balance over the day

In [10]:
for case in cases:
    fig = vp.power_balance(runs[(case, "EfficiencyBattery (losses)", 300)],
                           title=f"Active power balance, {case.lower()} (5-min steps, battery with losses)")
    fig.show()
    vp.save(fig, OUT / f"fig_power_balance_{case.split()[0].lower()}")

### 3.5 Battery models and step size (stress case)
`SimpleBattery` has no upper SOC limit and no losses; the `EfficiencyBattery` stops at 90 % and loses 5 % per direction.

In [11]:
soc_series = {}
for model_name in BATTERY_MODELS:
    for step in (900, 60):
        soc_series[f"{model_name}, {step // 60}-min steps"] = (
            runs[("Stress case", model_name, step)]["soc"] * 100)
fig = vp.line_comparison(soc_series, title="Battery state of charge, stress case",
                         y_title="Battery state of charge (%)")
fig.show()
vp.save(fig, OUT / "fig_soc_stress")

In [12]:
step_eff = closure.xs("Stress case", level="Case")[["Grid import", "Grid export", "Battery losses"]]
vp.format_table(step_eff, 2)

### 3.6 Wall time

In [13]:
fig = vp.line_comparison(
    {f"{c}, {m}": walls[(walls.Case == c) & (walls["Battery model"] == m)]
        .set_index("Steps")["Wall time (ms)"]
     for c in cases for m in BATTERY_MODELS},
    title="Vessim wall time for one simulated day", y_title="Wall time (ms)",
    x_title="Number of simulation steps", shape="linear")
fig.update_traces(mode="lines+markers", marker=dict(size=8))
fig.update_xaxes(type="log", tickvals=[96, 288, 1440],
                 ticktext=["96 (15 min)", "288 (5 min)", "1 440 (1 min)"], minor=dict(showgrid=False))
fig.update_yaxes(type="log", tickvals=[5, 10, 20, 50, 100], minor=dict(showgrid=False))
fig.show()
per_step_us = (walls["Wall time (ms)"] / walls["Steps"] * 1e3)
print(f"Wall time per step: {per_step_us.min():.0f}–{per_step_us.max():.0f} µs")

Wall time per step: 52–101 µs


### 3.7 Experiment record on disk

In [14]:
sorted(p.name for p in (OUT / "record").iterdir())

['experiment.json', 'timeseries.csv']

## 4. Evaluation

- **Toolchain works.** Vessim 0.16.0 (with mosaik extra), the helper package and the cached edge
  data load in `.venv`; the edge study has its own environment for generating reference data.
- **Energy is conserved.** The power balance residual is at floating-point level at every step for
  all 12 runs, and the daily closure error is ~0 kWh. Battery losses close exactly: 0 kWh for
  `SimpleBattery`, 5 % per direction for the `EfficiencyBattery`.
- **The built-in recorder metrics are correct**: grid import/export, produced and consumed energy
  and peak import match an independent integration of the time series.
- **All energy paths work** in the stress case: charging from surplus, a full battery, grid export
  and later discharge.
- **Step size does not change the daily totals** in either case (inputs are piecewise constant
  over 15 min and the greedy policy clips to the energy-feasible range, so a battery reaching its
  limit mid-interval stores the same energy). Only the time resolution of the SOC trajectory
  differs (see 3.5).
- **Speed.** Roughly 50–130 µs per step: 96 steps ≈ 5–13 ms, 1 440 steps ≈ 70–120 ms for one
  day (small runs are dominated by timing noise). The cost scales linearly with the number of steps.
- **Observation.** Vessim's built-in `SimpleBattery` charges up to 100 % SOC because it has no
  upper SOC limit — relevant when comparing with our 15–90 % window (see E01).